## Loading wind_power_per_bidzone dataset
This dataset contains hourly wind power production values aggregated at the bidding zone level. Each row is indexed by a timestamp, and each column corresponds to a Nordic bidding zone (e.g., ELSPOT NO1–NO4). The values represent the total wind power generated in each zone at that hour, and this dataset is typically used as the target variable for forecasting at the regional/bid-zone scale.

In [1]:
import pandas as pd

wind_power_per_bidzone = pd.read_parquet("../src/raw_data/wind_power_per_bidzone.parquet")
wind_power_per_bidzone.head()

,ELSPOT NO1,ELSPOT NO2,ELSPOT NO3,ELSPOT NO4
2020-01-01 00:00:00,149.285262,434.753864,497.143482,152.830002
2020-01-01 01:00:00,152.634024,395.572485,471.916899,144.440344
2020-01-01 02:00:00,151.163256,355.408523,399.325698,143.960894
2020-01-01 03:00:00,150.223341,375.316902,367.662849,134.310523
2020-01-01 04:00:00,157.415142,412.949107,399.627337,88.573315


## Loading windparks_bindzone dataset
This dataset provides metadata for wind parks mapped to bidding zones. Each row represents a single wind park and includes its associated bidding area, substation (wind park) name, maximum installed operating power, production start date, and a unique EIC identifier. The dataset is used to link individual wind parks to regional bidding zones and to support aggregation, capacity analysis, and consistency checks between meteorological inputs and zonal wind power production.

In [2]:
windparks_bindzone = pd.read_csv("../src/raw_data/windparks_bidzone.csv")
windparks_bindzone.head()

,bidding_area,substation_name,operating_power_max,prod_start_new,eic_code
0,ELSPOT NO1,Engerfjellet,52.8,2022-09-19 13:00:00,50WP00000002158R
1,ELSPOT NO1,Hån Vindpark,21.0,2022-10-26 22:00:00,50WP000000022251
2,ELSPOT NO1,Kjølberget,55.9,2020-07-07 09:00:00,50WP00000002085S
3,ELSPOT NO1,Marker Vindpark,54.0,2020-01-01 00:00:00,50WP00000002048Y
4,ELSPOT NO1,Raskiftet,111.6,2020-01-01 00:00:00,50WP00000001718K


In [3]:
windparks_bindzone["region"] = windparks_bindzone["bidding_area"].str.replace("ELSPOT ", "", regex=False).str.strip()
windparks_bindzone["prod_start_new"] = pd.to_datetime(windparks_bindzone["prod_start_new"], errors="coerce")
windparks_bindzone = windparks_bindzone.drop(columns="bidding_area")

In [4]:
windparks_bindzone.head()

,substation_name,operating_power_max,prod_start_new,eic_code,region
0,Engerfjellet,52.8,2022-09-19 13:00:00,50WP00000002158R,NO1
1,Hån Vindpark,21.0,2022-10-26 22:00:00,50WP000000022251,NO1
2,Kjølberget,55.9,2020-07-07 09:00:00,50WP00000002085S,NO1
3,Marker Vindpark,54.0,2020-01-01 00:00:00,50WP00000002048Y,NO1
4,Raskiftet,111.6,2020-01-01 00:00:00,50WP00000001718K,NO1


In [5]:
print(f"These are the following windparks in meta data: {windparks_bindzone["substation_name"].unique().tolist()}")

These are the following windparks in meta data: ['Engerfjellet', 'Hån Vindpark', 'Kjølberget', 'Marker Vindpark', 'Raskiftet', 'Songkjølen', 'Bjerk_VK Vindpark', 'Buheii Vindpark', 'Egersund Vindkrv', 'Gismarvik Vindpark', 'Håvik', 'Høg Jæren', 'Lista VK', 'Midtfjellet', 'Måkaknuten', 'Ramslandsvågen', 'Skudeneshavn', 'Stokkeland', 'Storøy vindpark', 'Svåheia', 'Tellenes', 'Tindafjellet', 'Tonst. Vindpark', 'Tysvær Vindpark', 'Vardafjell', 'Øie', 'Bessakerfjellet', 'Einarsdalen', 'Frøya Vindpark', 'Geitfjellet', 'Guleslettene Vindpark', 'Haraheia', 'Haram Kraft', 'Harbaksfjellet', 'Hennøy', 'Hitra', 'Hundhammerfjelle', 'Kvenndalsfjellet', 'Lutelandet', 'Mehuken', 'Okla Vindkraftverk', 'Orkdal', 'Sandøy Vindkraft', 'Skomakerfjellet', 'Smøla', 'Stokkfjellet', 'Storheia', 'Sørmarkfjellet', 'Valsn_Vimle', 'Valsneset', 'Ytre Vikna', 'Dønnesfjord Vind', 'Fakken', 'Hamnefjell', 'Havøygavlen', 'Hinnøy', 'Kjøllefjord vindpark', 'Kvitfjell vindpark', 'Nygårdsfjellet', 'Raggovidda', 'Raudfjell Vi

In [6]:
forecast_now_df = pd.read_parquet("../src/processed_data/notebook_data/met_forecast_nowcast_merged.parquet")

In [7]:
print(f"These are the following windparks in forecast: {forecast_now_df["sid_clean"].unique().tolist()}")

These are the following windparks in forecast: ['Engerfjellet', 'Marker Vindpark', 'Kjølberget', 'Songkjølen', 'Hån Vindpark', 'Raskiftet', 'Stokkeland', 'Måkaknuten', 'Buheii Vindpark', 'Vardafjell', 'Bjerk_VK Vindpark', 'Tonst', 'Midtfjellet', 'Tindafjellet', 'Storøy vindpark', 'Gismarvik Vindpark', 'Øie', 'Skudeneshavn', 'Tysvær Vindpark', 'Lista VK', 'Svåheia', 'Egersund Vindkrv', 'Tellenes', 'Høg Jæren', 'Smøla', 'Sørmarkfjellet', 'Ytre Vikna', 'Geitfjellet', 'Storheia', 'Hundhammerfjelle', 'Valsneset', 'Haraheia', 'Oklaverk', 'Stokkfjellet', 'Hitra', 'Bessakerfjellet', 'Frøya Vindpark', 'Einarsdalen', 'Haram Kraft', 'Mehuken', 'Lutelandet', 'Hennøy', 'Kvenndalsfjellet', 'Harbaksfjellet', 'Guleslettene Vindpark', 'Skomakerfjellet', 'Sandøy', 'Valsn_Vimle', 'Øyfjell1', 'Fakken', 'Øyfjell2', 'Nygårdsfjellet', 'Raggovidda', 'Kjøllefjord vindpark', 'Sørfj', 'Dønnesfjord Vind', 'Kvitfjell vindpark', 'Ånstadblåheia', 'Havøygavlen', 'Raudfjell Vindpark', 'Hamnefjell']


In [8]:
name_map = {
    "Tonst": "Tonst. Vindpark",
    "Oklaverk": "Okla Vindkraftverk",
    "Sandøy": "Sandøy Vindkraft",
    "Sørfj": "Sørfj. Vindkraft"
}


forecast_now_df["sid_clean"] = forecast_now_df["sid_clean"].replace(name_map)

In [9]:
left_keys = set(forecast_now_df["sid_clean"].dropna().unique())
right_keys = set(windparks_bindzone["substation_name"].dropna().unique())

print("parks in forecast_now_df but not in metadata:", len(left_keys - right_keys))
print("parks in metadata but not in forecast_now_df:", len(right_keys - left_keys))


parks in forecast_now_df but not in metadata: 0
parks in metadata but not in forecast_now_df: 4


In [10]:
meta_now_forecast_df = forecast_now_df.merge(
    windparks_bindzone[["substation_name", "region", "operating_power_max", "prod_start_new"]],
    left_on="sid_clean",
    right_on="substation_name",
    how="left"
).drop(columns=["substation_name"])

In [11]:
meta_now_forecast_df.head()

,time_ref,time,sid_clean,lt,ws10m_mean,ws10m_std,rh2m_mean,rh2m_std,t2m_mean,t2m_std,...,mslp_std,air_temperature_2m,air_pressure_at_sea_level,relative_humidity_2m,precipitation_amount,wind_speed_10m,wind_direction_10m,region,operating_power_max,prod_start_new
0,2020-02-15 12:00:00,2020-02-15 12:00:00,Engerfjellet,0,2.139608,0.448171,0.823404,0.051589,276.379494,0.395486,...,38.879500,275.69696,100518.625,0.844108,0.0,2.051558,193.094101,NO1,52.8,2022-09-19 13:00:00
1,2020-02-15 12:00:00,2020-02-15 13:00:00,Engerfjellet,1,3.384011,0.476801,0.825629,0.047831,276.422947,0.375587,...,21.675226,275.69696,100518.625,0.844108,0.0,2.051558,193.094101,NO1,52.8,2022-09-19 13:00:00
2,2020-02-15 12:00:00,2020-02-15 14:00:00,Engerfjellet,2,3.116065,0.380970,0.847620,0.059136,276.095935,0.374416,...,44.229292,275.69696,100518.625,0.844108,0.0,2.051558,193.094101,NO1,52.8,2022-09-19 13:00:00
3,2020-02-15 12:00:00,2020-02-15 15:00:00,Engerfjellet,3,2.856627,0.388535,0.886108,0.062451,275.603674,0.339881,...,49.885909,275.69696,100518.625,0.844108,0.0,2.051558,193.094101,NO1,52.8,2022-09-19 13:00:00
4,2020-02-15 12:00:00,2020-02-15 16:00:00,Engerfjellet,4,3.303811,0.599790,0.942954,0.047610,274.942599,0.471217,...,66.165161,275.69696,100518.625,0.844108,0.0,2.051558,193.094101,NO1,52.8,2022-09-19 13:00:00


In [12]:
import numpy as np

wcol = "operating_power_max"
keys = ["region", "time_ref", "time", "lt"]

wmean_cols = [
    "ws10m_mean","ws10m_std","rh2m_mean","rh2m_std","t2m_mean","t2m_std",
    "g10m_mean","g10m_std","mslp_mean","mslp_std",
    "air_temperature_2m","air_pressure_at_sea_level","relative_humidity_2m",
    "wind_speed_10m"
]

tmp = meta_now_forecast_df[keys + [wcol] + wmean_cols].copy()

# weight sum per group
w_sum = tmp.groupby(keys)[wcol].sum().rename("capacity_total")

# weighted numerator sums
for c in wmean_cols:
    tmp[c] = pd.to_numeric(tmp[c], errors="coerce")
    tmp[c] = tmp[c] * tmp[wcol]

num_sum = tmp.groupby(keys)[wmean_cols].sum()

region_df = (num_sum.div(w_sum, axis=0)
                    .reset_index()
                    .merge(w_sum.reset_index(), on=keys, how="left"))

In [13]:
prec = meta_now_forecast_df.assign(precipitation_amount=pd.to_numeric(meta_now_forecast_df["precipitation_amount"], errors="coerce")) \
         .groupby(keys)["precipitation_amount"].sum().reset_index()

region_df = region_df.merge(prec, on=keys, how="left")

In [14]:
def add_circular_mean(region_df, df, colname, outname):
    ang = pd.to_numeric(df[colname], errors="coerce")
    w = pd.to_numeric(df[wcol], errors="coerce")
    m = ang.notna() & w.notna()
    sub = df.loc[m, keys].copy()
    rad = np.deg2rad(ang[m].values)

    sub["_sin"] = np.sin(rad) * w[m].values
    sub["_cos"] = np.cos(rad) * w[m].values

    agg = sub.groupby(keys)[["_sin","_cos"]].sum().reset_index()
    agg[outname] = (np.rad2deg(np.arctan2(agg["_sin"], agg["_cos"])) % 360)
    agg = agg.drop(columns=["_sin","_cos"])

    return region_df.merge(agg, on=keys, how="left")

region_df = add_circular_mean(region_df, meta_now_forecast_df, "wd10m_mean", "wd10m_mean")
region_df = add_circular_mean(region_df, meta_now_forecast_df, "wind_direction_10m", "wind_direction_10m")
region_df["region"].value_counts()


region
NO1    112692
NO2    112692
NO3    112692
NO4    112692
Name: count, dtype: int64

In [15]:
region_df.head()

,region,time_ref,time,lt,ws10m_mean,ws10m_std,rh2m_mean,rh2m_std,t2m_mean,t2m_std,...,mslp_mean,mslp_std,air_temperature_2m,air_pressure_at_sea_level,relative_humidity_2m,wind_speed_10m,capacity_total,precipitation_amount,wd10m_mean,wind_direction_10m
0,NO1,2020-02-15 12:00:00,2020-02-15 12:00:00,0,2.885023,0.564210,0.849861,0.049327,275.620253,0.453982,...,100523.391057,39.606556,275.814952,100545.634126,0.861225,2.652674,405.7,0.001116,218.406386,214.646028
1,NO1,2020-02-15 12:00:00,2020-02-15 13:00:00,1,3.233010,0.406024,0.838769,0.050891,275.693526,0.425750,...,100553.875387,28.523728,275.814952,100545.634126,0.861225,2.652674,405.7,0.001116,217.522286,214.646028
2,NO1,2020-02-15 12:00:00,2020-02-15 14:00:00,2,3.101949,0.434790,0.862924,0.055345,275.377810,0.414945,...,100541.651009,43.219588,275.814952,100545.634126,0.861225,2.652674,405.7,0.001116,204.505754,214.646028
3,NO1,2020-02-15 12:00:00,2020-02-15 15:00:00,3,2.913593,0.367008,0.908355,0.053968,274.832810,0.385139,...,100491.005989,55.597789,275.814952,100545.634126,0.861225,2.652674,405.7,0.001116,190.803449,214.646028
4,NO1,2020-02-15 12:00:00,2020-02-15 16:00:00,4,3.225136,0.475355,0.961101,0.036191,274.208863,0.461787,...,100435.775875,63.683048,275.814952,100545.634126,0.861225,2.652674,405.7,0.001116,183.557128,214.646028


In [16]:
wind_power_per_bidzone = wind_power_per_bidzone.reset_index().rename(columns={"index": "time"})

wind_power_long = wind_power_per_bidzone.melt(
    id_vars="time",
    var_name="region",
    value_name="power_MW"
)

# clean region labels: "ELSPOT NO1" → "NO1"
wind_power_long["region"] = wind_power_long["region"].str.replace(
    "ELSPOT ", "", regex=False
)

wind_power_long.head()

,time,region,power_MW
0,2020-01-01 00:00:00,NO1,149.285262
1,2020-01-01 01:00:00,NO1,152.634024
2,2020-01-01 02:00:00,NO1,151.163256
3,2020-01-01 03:00:00,NO1,150.223341
4,2020-01-01 04:00:00,NO1,157.415142


In [17]:
final_df = region_df.merge(
    wind_power_long,
    on=["region", "time"],
    how="left"
)

In [18]:
final_df.to_parquet("../src/processed_data/notebook_data/alligned_data.parquet")